# 09 — Uncertainty-Aware Routing

Route queries to smaller models when confident, larger models when uncertain.

## Routing to Larger Models and Abstention

**Cascading / routing** systems use cheap models for easy queries and expensive models only for hard ones:
1. Run a small, fast model on the query
2. Estimate uncertainty (logit confidence, semantic entropy, or external classifier)
3. If uncertainty < threshold → return small model's answer
4. If uncertainty ≥ threshold → escalate to larger model

**Design choices**:
- *Routing signal*: what triggers escalation? Confidence score, uncertainty estimate, query complexity classifier
- *Threshold*: calibrate on a held-out set to achieve target coverage
- *Cost model*: small model cost vs large model cost vs user experience impact

**Abstention** is a special case: the model says 'I don't know' and routes to a human. Appropriate when:
- No models achieve acceptable accuracy
- The query is outside the deployment scope
- High-stakes decisions require human review (e.g., medical, legal)

**FrugalGPT** (Chen et al., 2023) formalises LLM cascades: select the cheapest sufficient model for each query, reducing costs by 98% on some benchmarks while matching GPT-4 accuracy.

In [ ]:
# Uncertainty-based model router
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

class SmallModel:
    """Cheap, fast, ~70% accuracy. Returns confidence score."""
    def predict(self, q_id):
        np.random.seed(q_id)
        conf = np.random.beta(2, 2)  # spread confidence
        correct = np.random.random() < (0.3 + 0.6 * conf)
        return conf, int(correct), cost=0.001  # $0.001 per query

class LargeModel:
    """Expensive, slower, ~90% accuracy. Always returns high confidence."""
    def predict(self, q_id):
        np.random.seed(q_id + 10000)
        correct = np.random.random() < 0.90
        return 0.9, int(correct), 0.10  # $0.10 per query

# Fix the class definition to not use keyword in wrong position
class SmallModel2:
    def predict(self, q_id):
        np.random.seed(q_id)
        conf = np.random.beta(2, 2)
        correct = int(np.random.random() < (0.3 + 0.6 * conf))
        return conf, correct, 0.001

class LargeModel2:
    def predict(self, q_id):
        np.random.seed(q_id + 10000)
        correct = int(np.random.random() < 0.90)
        return 0.9, correct, 0.10

small = SmallModel2()
large = LargeModel2()

def routing_policy(conf, threshold):
    return 'small' if conf >= threshold else 'large'

# Evaluate at different routing thresholds
thresholds = np.linspace(0, 1, 20)
n_queries = 1000

results = []
for thresh in thresholds:
    total_cost = 0
    total_correct = 0
    large_calls = 0

    for q_id in range(n_queries):
        s_conf, s_correct, s_cost = small.predict(q_id)
        if routing_policy(s_conf, thresh) == 'small':
            total_cost += s_cost
            total_correct += s_correct
        else:
            _, l_correct, l_cost = large.predict(q_id)
            total_cost += s_cost + l_cost  # still paid for small model call
            total_correct += l_correct
            large_calls += 1

    results.append({
        'threshold': thresh,
        'accuracy': total_correct / n_queries,
        'avg_cost': total_cost / n_queries,
        'large_fraction': large_calls / n_queries,
    })

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

accs = [r['accuracy'] for r in results]
costs = [r['avg_cost'] for r in results]
fracs = [r['large_fraction'] for r in results]

ax1.plot(fracs, accs, 'o-', color='steelblue')
ax1.axhline(0.9, color='tomato', linestyle='--', label='Large model only')
ax1.axhline(np.array([r['accuracy'] for r in results if r['threshold'] == 1.0][0] if any(r['threshold'] == 1.0 for r in results) else 0.7), color='orange', linestyle='--', label='Small model only')
ax1.set_xlabel('Fraction of queries to large model')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs Escalation Rate')
ax1.legend()

ax2.plot(costs, accs, 'o-', color='purple')
ax2.set_xlabel('Average cost per query ($)')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy vs Cost (Pareto frontier)')

plt.suptitle('Uncertainty-Aware Model Routing')
plt.tight_layout()
plt.savefig('/tmp/routing.png', dpi=100, bbox_inches='tight')
plt.show()

best = max(results, key=lambda r: r['accuracy'] / (r['avg_cost'] + 0.01))
print(f'Optimal threshold: {best["threshold"]:.2f}')
print(f'  Accuracy: {best["accuracy"]:.3f}')
print(f'  Cost: ${best["avg_cost"]:.4f}/query')
print(f'  Large model calls: {best["large_fraction"]:.1%}')